In [10]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from core.simulator import Simulator

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from importlib import reload

import core.simulator, core.geometry, core.membrane
import core.concentration, core.registry, core.ficks, core.analysis

reload(core.simulator); reload(core.geometry); reload(core.membrane)
reload(core.concentration); reload(core.registry)
reload(core.ficks); reload(core.analysis)

from core.simulator     import Simulator
from core.geometry      import BoxGeometry
from core.membrane      import Membrane
from core.concentration import ConcentrationField, FirstPassageTracker
from core.registry      import Registry
from core.ficks         import FicksSolver
from core.analysis      import summarise, print_summary

# load drug + tissue
drug   = Registry.load_drug("ibuprofen")
tissue = Registry.load_tissue("blood_brain_barrier")
D      = Registry.compute_D_eff(drug, tissue)

# setup simulation
box      = BoxGeometry([(-tissue.box_size/2, tissue.box_size/2)] * 3)
membrane = Membrane(axis=2, position=0.0, permeability=tissue.permeability)
conc     = ConcentrationField(box_size=tissue.box_size, n_voxels=20)
tracker  = FirstPassageTracker(axis=2, threshold=tissue.box_size * 0.4, dt=0.001)

sim = Simulator(
    n_particles=500, D=D, dt=0.001,
    geometry=box, membrane=membrane,
    concentration_field=conc, fpt_tracker=tracker
)

# start particles on left side
sim.positions[:, 2] = np.random.uniform(-tissue.box_size/2, 0, 500)
sim.history = [sim.positions.copy()]

# run simulation + ensemble
sim.run(n_steps=3000)
result = sim.run_ensemble(n_steps=1000, n_runs=50)

print("Done — now building summary...")

TypeError: FirstPassageTracker.__init__() missing 1 required positional argument: 'n_particles'

In [ ]:
from core.umhm import umhm

ImportError: cannot import name 'umhm' from 'core.umhm' (c:\Users\Nandan Thakare\OneDrive\Documents\drugnandan\notebooks\..\core\umhm.py)

In [2]:
 
import numpy as np
from scipy import stats
from dataclasses import dataclass
from typing import Optional

GREEN  = "\033[92m"
RED    = "\033[91m"
YELLOW = "\033[93m"
RESET  = "\033[0m"
BOLD   = "\033[1m"
 
def passed(msg):  print(f"  {GREEN}✓ PASS{RESET}  {msg}")
def failed(msg):  print(f"  {RED}✗ FAIL{RESET}  {msg}")
def warn(msg):    print(f"  {YELLOW}⚠ WARN{RESET}  {msg}")
def header(msg):  print(f"\n{BOLD}{msg}{RESET}\n" + "─" * 55)
 
PASS = True   # global flag — set False on any failure
 
def check(condition, pass_msg, fail_msg):
    global PASS
    if condition:
        passed(pass_msg)
    else:
        failed(fail_msg)
        PASS = False
 
np.random.seed(42)
D   = 1e-12   # m²/s
dt  = 1e-4    # s
N   = 2000    # particles
T   = 500     # steps
 
positions = np.zeros((N, 3))
origin    = positions.copy()
msds      = []
 
for _ in range(T):
    sigma      = np.sqrt(2 * D * dt)
    positions += np.random.normal(0, sigma, size=(N, 3))
    msd        = np.mean(np.sum((positions - origin)**2, axis=1))
    msds.append(msd)
 
times       = np.arange(1, T + 1) * dt
msds        = np.array(msds)
theory      = 6 * D * times
 
# linear regression on MSD vs time — slope should be 6D
slope, intercept, r, p, se = stats.linregress(times, msds)
expected_slope = 6 * D
 
rel_err = abs(slope - expected_slope) / expected_slope
 
check(
    rel_err < 0.05,
    f"MSD slope = {slope:.3e}, theory = {expected_slope:.3e}  (err={rel_err*100:.1f}%)",
    f"MSD slope = {slope:.3e}, theory = {expected_slope:.3e}  (err={rel_err*100:.1f}%) — too large"
)
check(
    r**2 > 0.999,
    f"MSD R² = {r**2:.6f}  (linear as expected for free diffusion)",
    f"MSD R² = {r**2:.6f}  — non-linear, something wrong"
)
 
# Confirm 2D formula would be WRONG
wrong_theory = 4 * D * times
slope_2d, _, _, _, _ = stats.linregress(times, msds / wrong_theory)
check(
    abs(slope_2d - 1.0) > 0.2,
    f"4Dt (2D formula) gives wrong fit — confirms 6Dt is correct for 3D",
    f"4Dt fits just as well — dimension check ambiguous"
)
 

  ✓ PASS  MSD slope = 6.129e-12, theory = 6.000e-12  (err=2.1%)
  ✓ PASS  MSD R² = 0.999780  (linear as expected for free diffusion)
  ✓ PASS  4Dt (2D formula) gives wrong fit — confirms 6Dt is correct for 3D


In [4]:
# run this one-liner in your project
from core.simulator import Simulator
import numpy as np

sim = Simulator(n_particles=1000, D=1e-12, dt=1e-4)
sim.run(500)
msd = sim.get_msd()
t = np.arange(501) * 1e-4
slope = np.polyfit(t, msd, 1)[0]
print(f"slope = {slope:.3e}, theory 6D = {6e-12:.3e}, ratio = {slope/6e-12:.3f}")

ModuleNotFoundError: No module named 'core'

In [1]:
import yaml
from pathlib import Path

drug_dir = Path("../data/drugs")
null_files = []
empty_fields = []

for f in drug_dir.glob("*.yaml"):
    with open(f, encoding="utf-8") as file:
        data = yaml.safe_load(file)
    if data is None:
        null_files.append(f)
    elif data.get("molecular_weight") is None:
        empty_fields.append(f)

print(f"Completely null files:     {len(null_files)}")
print(f"Missing molecular_weight:  {len(empty_fields)}")
print(f"Total to clean:            {len(null_files) + len(empty_fields)}")

Completely null files:     0
Missing molecular_weight:  0
Total to clean:            0


In [2]:
import yaml
from pathlib import Path

drug_dir = Path("../data/drugs")

stats = {"total": 0, "has_mw": 0, "has_logp": 0, 
         "has_pka": 0, "has_pb": 0, "all_four": 0}

for f in drug_dir.glob("*.yaml"):
    with open(f, encoding="utf-8") as file:
        data = yaml.safe_load(file)
    if data is None:
        continue
    
    stats["total"] += 1
    has_mw   = data.get("molecular_weight") is not None
    has_logp = data.get("logP") is not None
    has_pka  = data.get("pKa") is not None
    has_pb   = data.get("protein_binding") is not None
    
    if has_mw:   stats["has_mw"]   += 1
    if has_logp: stats["has_logp"] += 1
    if has_pka:  stats["has_pka"]  += 1
    if has_pb:   stats["has_pb"]   += 1
    if all([has_mw, has_logp, has_pka, has_pb]): stats["all_four"] += 1

for k, v in stats.items():
    pct = f"({v/stats['total']*100:.1f}%)" if k != "total" else ""
    print(f"{k:<20} {v:>6} {pct}")

total                  9562 
has_mw                 9562 (100.0%)
has_logp               1475 (15.4%)
has_pka                 484 (5.1%)
has_pb                 1061 (11.1%)
all_four                162 (1.7%)


In [3]:
import yaml
from pathlib import Path

drug_dir = Path("../data/drugs")
deleted = 0
kept = 0

for f in drug_dir.glob("*.yaml"):
    with open(f, encoding="utf-8") as file:
        data = yaml.safe_load(file)
    
    if data is None:
        f.unlink()
        deleted += 1
        continue
    
    has_mw   = data.get("molecular_weight") is not None
    has_logp = data.get("logP") is not None
    has_pka  = data.get("pKa") is not None
    has_pb   = data.get("protein_binding") is not None
    
    if all([has_mw, has_logp, has_pka, has_pb]):
        kept += 1
    else:
        f.unlink()
        deleted += 1

print(f"Kept:    {kept}")
print(f"Deleted: {deleted}")

Kept:    162
Deleted: 9400


In [5]:
from pathlib import Path

must_have = ["ibuprofen", "aspirin", "acetaminophen", "diazepam", 
             "metformin", "atorvastatin", "acetylsalicylic_acid"]

drug_dir = Path("../data/drugs")
for name in must_have:
    exists = (drug_dir / f"{name}.yaml").exists()
    print(f"{'✅' if exists else '❌'} {name}")

✅ ibuprofen
❌ aspirin
✅ acetaminophen
✅ diazepam
❌ metformin
❌ atorvastatin
✅ acetylsalicylic_acid


In [6]:
from pathlib import Path

drug_dir = Path("../data/drugs")
drugs = sorted([f.stem for f in drug_dir.glob("*.yaml")])
print(f"Total: {len(drugs)}\n")
for d in drugs:
    print(d)

Total: 162

acemetacin
acetaminophen
acetazolamide
acetylsalicylic_acid
aciclovir
alendronic_acid
alprostadil
aminosalicylic_acid
amisulpride
amitriptyline
amphetamine
apomorphine
atenolol
atropine
azathioprine
azithromycin
bendroflumethiazide
benzthiazide
benzylpenicillin
betaxolol
bicalutamide
bupivacaine
buprenorphine
caffeine
cephalexin
chlorambucil
chlorhexidine
chloroquine
chlorothiazide
chlorphenamine
chlorpromazine
cimetidine
ciprofloxacin
clarithromycin
clonidine
cloxacillin
clozapine
codeine
colchicine
cyclobenzaprine
dapsone
desipramine
dexlansoprazole
dexmedetomidine
dexrazoxane
diazepam
diazoxide
diclofenac
diphenhydramine
dolutegravir
domperidone
dronabinol
emtricitabine
enalapril
erythromycin
etacrynic_acid
etodolac
fenoprofen
flecainide
flucytosine
fluorouracil
gefitinib
gemcitabine
glipizide
haloperidol
heroin
hexachlorophene
hexobarbital
hydrochlorothiazide
hydroflumethiazide
hyoscyamine
ibuprofen
imipramine
indapamide
indomethacin
isoniazid
itraconazole
ketoprofen
ke

In [1]:
import pandas as pd

hpa = pd.read_csv("../data/rna_tissue_consensus_data.tsv", sep="\t", nrows=5)
print(hpa.columns.tolist())
print(hpa.head())

FileNotFoundError: [Errno 2] No such file or directory: '../data/rna_tissue_consensus_data.tsv'

In [2]:
from pathlib import Path

data_dir = Path("../data")
for f in data_dir.iterdir():
    print(f.name)

BDB-mySQL_All_202604_dmp
drugbank.xml
drugs
normal_ihc_data.tsv
rna_tissue_consensus.tsv
tissues


In [3]:
import pandas as pd

hpa = pd.read_csv("../data/rna_tissue_consensus.tsv", sep="\t", nrows=5)
print(hpa.columns.tolist())
print(hpa.head())


['Gene', 'Gene name', 'Tissue', 'nTPM']
              Gene Gene name          Tissue  nTPM
0  ENSG00000000003    TSPAN6  adipose tissue  20.1
1  ENSG00000000003    TSPAN6   adrenal gland  13.3
2  ENSG00000000003    TSPAN6        amygdala   8.7
3  ENSG00000000003    TSPAN6        appendix   4.6
4  ENSG00000000003    TSPAN6   basal ganglia   8.4


In [6]:
import pandas as pd

hpa_full = pd.read_csv("../data/rna_tissue_consensus.tsv", sep="\t")

# check available tissues
print("Tissues in HPA:")
print(sorted(hpa_full['Tissue'].unique()))
print()

# check our transporter genes
transporters = ["ABCB1", "ABCG2", "SLCO1B1", "SLC22A1", "ABCC2"]
print("Transporter genes found:")
print(hpa_full[hpa_full['Gene name'].isin(transporters)][['Gene name', 'Tissue', 'nTPM']].to_string())

Tissues in HPA:
['adipose tissue', 'adrenal gland', 'amygdala', 'appendix', 'basal ganglia', 'blood vessel', 'bone marrow', 'breast', 'cerebellum', 'cerebral cortex', 'cervix', 'choroid plexus', 'colon', 'duodenum', 'endometrium', 'epididymis', 'esophagus', 'fallopian tube', 'gallbladder', 'heart muscle', 'hippocampal formation', 'hypothalamus', 'kidney', 'liver', 'lung', 'lymph node', 'midbrain', 'ovary', 'pancreas', 'parathyroid gland', 'pituitary gland', 'placenta', 'prostate', 'rectum', 'retina', 'salivary gland', 'seminal vesicle', 'skeletal muscle', 'skin', 'small intestine', 'smooth muscle', 'spinal cord', 'spleen', 'stomach', 'testis', 'thymus', 'thyroid gland', 'tongue', 'tonsil', 'urinary bladder', 'vagina']

Transporter genes found:
       Gene name                 Tissue   nTPM
21981      ABCC2         adipose tissue    1.2
21982      ABCC2          adrenal gland    0.4
21983      ABCC2               amygdala    0.1
21984      ABCC2               appendix    0.7
21985      

In [7]:
import yaml
import numpy as np
from pathlib import Path

# HPA tissue name → your tissue YAML name
TISSUE_MAP = {
    "liver":           "liver",
    "lung":            "lung",
    "skin":            "skin",
    "skeletal muscle": "muscle",
    "small intestine": "intestine",
    "cerebral cortex": "blood_brain_barrier",  # closest proxy for BBB
}

TRANSPORTERS = ["ABCB1", "ABCG2", "SLCO1B1", "SLC22A1", "ABCC2"]

# filter to our transporters
df = hpa_full[hpa_full['Gene name'].isin(TRANSPORTERS)].copy()

# compute expression_scale = nTPM / max nTPM for that gene
df['max_nTPM'] = df.groupby('Gene name')['nTPM'].transform('max')
df['expression_scale'] = (df['nTPM'] / df['max_nTPM'].replace(0, 1)).round(4)

# build nested dict: tissue → gene → expression_scale
results = {}
for hpa_tissue, yaml_tissue in TISSUE_MAP.items():
    subset = df[df['Tissue'] == hpa_tissue]
    results[yaml_tissue] = {}
    for _, row in subset.iterrows():
        results[yaml_tissue][row['Gene name']] = float(row['expression_scale'])

# print preview
for tissue, genes in results.items():
    print(f"\n{tissue}:")
    for gene, scale in genes.items():
        print(f"  {gene}: {scale}")


liver:
  ABCC2: 1.0
  ABCB1: 0.2716
  ABCG2: 0.405
  SLCO1B1: 1.0
  SLC22A1: 1.0

lung:
  ABCC2: 0.0044
  ABCB1: 0.0624
  ABCG2: 0.0577
  SLCO1B1: 0.0
  SLC22A1: 0.0007

skin:
  ABCC2: 0.0141
  ABCB1: 0.0146
  ABCG2: 0.006
  SLCO1B1: 0.0
  SLC22A1: 0.0007

muscle:
  ABCC2: 0.0088
  ABCB1: 0.0021
  ABCG2: 0.0036
  SLCO1B1: 0.0
  SLC22A1: 0.0015

intestine:
  ABCC2: 0.333
  ABCB1: 0.693
  ABCG2: 1.0
  SLCO1B1: 0.0
  SLC22A1: 0.0003

blood_brain_barrier:
  ABCC2: 0.0009
  ABCB1: 0.0905
  ABCG2: 0.1647
  SLCO1B1: 0.0
  SLC22A1: 0.0002


In [8]:
tissue_dir = Path("../data/tissues")

for tissue_name, gene_scales in results.items():
    path = tissue_dir / f"{tissue_name}.yaml"
    if not path.exists():
        print(f"skipping {tissue_name} — yaml not found")
        continue
    
    with open(path, encoding="utf-8") as f:
        data = yaml.safe_load(f)
    
    # add transporter expression scales
    data["transporter_expression"] = gene_scales
    data["transporter_expression_source"] = "Human Protein Atlas — rna_tissue_consensus nTPM, normalised to max expression"
    
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(data, f, allow_unicode=True, sort_keys=False)
    
    print(f"updated {tissue_name}.yaml")

updated liver.yaml
updated lung.yaml
updated skin.yaml
updated muscle.yaml
updated intestine.yaml
updated blood_brain_barrier.yaml


In [11]:
from importlib import reload
import core.registry
reload(core.registry)
from core.registry import Registry

liver = Registry.load_tissue("liver")
bbb   = Registry.load_tissue("blood_brain_barrier")

print("Liver transporter expression:")
for gene, scale in liver.transporter_expression.items():
    print(f"  {gene}: {scale}")

print("\nBBB transporter expression:")
for gene, scale in bbb.transporter_expression.items():
    print(f"  {gene}: {scale}")

print(f"\nP-gp in liver: {liver.get_expression_scale('ABCB1')}")
print(f"P-gp in BBB:   {bbb.get_expression_scale('ABCB1')}")
print(f"OATP1B1 in liver: {liver.get_expression_scale('SLCO1B1')}")
print(f"OATP1B1 in BBB:   {bbb.get_expression_scale('SLCO1B1')}")

Liver transporter expression:
  ABCC2: 1.0
  ABCB1: 0.2716
  ABCG2: 0.405
  SLCO1B1: 1.0
  SLC22A1: 1.0

BBB transporter expression:
  ABCC2: 0.0009
  ABCB1: 0.0905
  ABCG2: 0.1647
  SLCO1B1: 0.0
  SLC22A1: 0.0002

P-gp in liver: 0.2716
P-gp in BBB:   0.0905
OATP1B1 in liver: 1.0
OATP1B1 in BBB:   0.0


In [12]:
from importlib import reload
import core.registry
reload(core.registry)
from core.registry import Registry

drug   = Registry.load_drug("ibuprofen")
tissue = Registry.load_tissue("blood_brain_barrier")

pgp = Registry.make_transporter_for_tissue(
    gene="ABCB1", tissue=tissue, drug=drug,
    Km_um=10.0, Vmax_ms=3e-9
)
print(pgp)
print(f"expression_scale: {pgp.params.expression_scale}")
print(f"free_fraction:    {pgp.params.free_fraction}")

ActiveTransporter(P-gp, efflux, axis=2, zone=left)
expression_scale: 0.0905
free_fraction:    0.010000000000000009


In [2]:
import reportlab
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.enums import TA_LEFT, TA_CENTER

W, H = A4

doc = SimpleDocTemplate(
    "/mnt/user-data/outputs/Study_Guide_FDA.pdf",
    pagesize=A4,
    leftMargin=18*mm, rightMargin=18*mm,
    topMargin=18*mm, bottomMargin=18*mm,
)

# ── Colour palette ──────────────────────────────────────────────────────────
C = {
    "purple": colors.HexColor("#534AB7"),
    "teal":   colors.HexColor("#0F6E56"),
    "blue":   colors.HexColor("#185FA5"),
    "coral":  colors.HexColor("#993C1D"),
    "amber":  colors.HexColor("#854F0B"),
    "green":  colors.HexColor("#3B6D11"),
    "pink":   colors.HexColor("#993556"),
    "red":    colors.HexColor("#A32D2D"),
    "gray":   colors.HexColor("#5F5E5A"),
    "bg":     colors.HexColor("#F7F6F3"),
    "white":  colors.white,
    "text":   colors.HexColor("#1A1A1A"),
    "muted":  colors.HexColor("#5A5A5A"),
}

styles = getSampleStyleSheet()

def S(name, **kw):
    return ParagraphStyle(name, **kw)

cover_title = S("CoverTitle", fontName="Helvetica-Bold", fontSize=28, textColor=C["white"],
                leading=34, alignment=TA_CENTER, spaceAfter=6)
cover_sub   = S("CoverSub",   fontName="Helvetica",      fontSize=13, textColor=colors.HexColor("#DDD8FF"),
                leading=18, alignment=TA_CENTER)
mod_title   = S("ModTitle",   fontName="Helvetica-Bold", fontSize=13, textColor=C["white"],
                leading=16, spaceAfter=0, spaceBefore=0)
mod_file    = S("ModFile",    fontName="Helvetica",      fontSize=8,  textColor=colors.HexColor("#E0E0E0"),
                leading=10, spaceAfter=0)
topic_head  = S("TopicHead",  fontName="Helvetica-Bold", fontSize=10, textColor=C["text"],
                leading=13, spaceBefore=8, spaceAfter=2)
sub_item    = S("SubItem",    fontName="Helvetica",      fontSize=9,  textColor=C["muted"],
                leading=13, leftIndent=12, spaceBefore=0, spaceAfter=0)
page_title  = S("PageTitle",  fontName="Helvetica-Bold", fontSize=16, textColor=C["text"],
                leading=20, spaceBefore=0, spaceAfter=4)

# ── Data ────────────────────────────────────────────────────────────────────
modules = [
    {
        "label": "Module 1 — Data Fundamentals",
        "file": "Module_1_FDA.pptx",
        "color": "purple",
        "topics": [
            ("What is Data?", ["Definition and properties", "Raw vs processed data", "Sources: humans, machines, human-machine"]),
            ("Data Objects & Attributes", ["Attribute types overview", "Nominal attributes", "Binary attributes", "Ordinal attributes", "Numeric attributes (interval & ratio)", "Discrete vs continuous attributes", "Properties of attribute values (distinctness, order, addition, multiplication)"]),
            ("Characteristics of Datasets", ["Dimensionality & curse of dimensionality", "Sparsity", "Resolution"]),
            ("Types of Datasets", ["Record data", "Data matrix", "Document data (Bag-of-Words)", "Transaction data", "Graph-based data (nodes & edges)", "Sequential data", "Sequence data", "Time-series data", "Spatial data", "Components of time series (trend, seasonality, cyclical, irregular)"]),
        ]
    },
    {
        "label": "Module 2 — Data & Text Visualization",
        "file": "Module_2_FDA.pptx",
        "color": "teal",
        "topics": [
            ("What is Data Visualization?", ["Definition and importance", "Benefits of visualization", "Use cases: sales, healthcare, finance, logistics, politics"]),
            ("Types of Analysis", ["Univariate analysis", "Bivariate analysis", "Multivariate analysis"]),
            ("Structured Data Charts", ["Bar chart (horizontal, stacked, grouped)", "Pie chart", "Histogram", "Stacked bar chart", "Box plot (box-and-whisker)", "Scatter plot", "Heat map", "Line graph", "Dual axis (combo) chart", "Ring/doughnut chart", "Venn diagram"]),
            ("Network & Hierarchical Charts", ["Node-link diagram", "Tree maps / tree diagrams", "Matrix chart"]),
            ("Geospatial Charts", ["Flow map", "Density map"]),
            ("Text Visualization", ["Word count (quantitative analysis)", "Word tree", "Word clouds (tag clouds)"]),
            ("7 Stages of Data Visualization", ["Acquire", "Parse", "Filter", "Mine", "Represent", "Refine", "Interact"]),
            ("Visualization Tools", ["Tableau", "Microsoft Power BI", "Excel", "Python (Matplotlib, Seaborn)"]),
        ]
    },
    {
        "label": "Module 3 — Descriptive Statistics",
        "file": "Descriptive_Statistics_new.pptx",
        "color": "blue",
        "topics": [
            ("Statistics Basics", ["Population vs sample", "Key terminologies (variable, data, outlier, confidence interval, hypothesis, correlation, regression, ANOVA, Chi-square)", "Types: descriptive vs inferential statistics"]),
            ("Grouped & Ungrouped Data", ["Raw / ungrouped data", "Frequency distribution / grouped data"]),
            ("Measures of Central Tendency", ["Arithmetic mean (simple & weighted)", "Properties of arithmetic mean", "Median (ungrouped & grouped)", "Mode (ungrouped & grouped)", "Empirical relationship: Mode = 3 Median - 2 Mean", "Symmetric, positively & negatively skewed distributions", "Geometric mean (ungrouped & grouped)", "Harmonic mean (ungrouped & grouped)", "Relation between AM, GM, HM"]),
            ("Outliers (introduction)", ["Definition", "IQR method to identify outliers"]),
        ]
    },
    {
        "label": "Quartiles",
        "file": "4__Quartile_new.pptx",
        "color": "coral",
        "topics": [
            ("Quartiles", ["Definition and purpose (dividing data into 4 equal parts)", "Q1, Q2, Q3 formulas", "Calculating quartiles for odd-sized datasets", "Calculating quartiles for even-sized datasets (interpolation)", "Quartiles for grouped / class data"]),
            ("Interquartile Range (IQR)", ["Definition: IQR = Q3 - Q1", "Spread of middle 50%", "Small vs large IQR interpretation"]),
        ]
    },
    {
        "label": "Dispersion",
        "file": "5__Dispersion1_new.pptx",
        "color": "amber",
        "topics": [
            ("Introduction to Dispersion", ["Definition: scatter, spread, variation", "Homogeneous vs heterogeneous data", "Absolute vs relative measures"]),
            ("Absolute Measures of Dispersion", ["Range (ungrouped & grouped) - advantages & disadvantages", "Quartile Deviation / Semi-IQR (individual & grouped)", "Mean Deviation (from mean & median, ungrouped & grouped)", "Standard Deviation (sigma) - formula, examples", "Variance - definition, relation to SD"]),
            ("Skewness", ["Symmetric vs asymmetric distribution", "Positive skew: Mean > Median > Mode", "Negative skew: Mean < Median < Mode"]),
            ("Measures of Skewness", ["Karl-Pearson's coefficient (mode & median versions)", "Bowley's coefficient of skewness", "Coefficient of skewness based on moments (beta-1)"]),
            ("Kurtosis", ["Definition and purpose", "Leptokurtic (beta-2 > 3)", "Mesokurtic / normal (beta-2 = 3)", "Platykurtic (beta-2 < 3)", "Karl Pearson's measure of kurtosis (beta-2, gamma-2)"]),
        ]
    },
    {
        "label": "Relative Measures of Dispersion",
        "file": "Realtive_measure_of_disperssion.pptx",
        "color": "green",
        "topics": [
            ("Relative Measures (Coefficients)", ["Coefficient of Range", "Coefficient of Quartile Deviation", "Coefficient of Mean Deviation", "Coefficient of Standard Deviation", "Coefficient of Variation (C.V.)"]),
            ("Moments in Statistics", ["Definition of rth order moment", "First moment (m1 = 0)", "Second moment (m2 = Variance)", "Third moment (m3 = Skewness)", "Fourth moment (m4 = Kurtosis)"]),
            ("Skewness (revisited)", ["Interpretation of skewness values", "Degree of skewness (slight, moderate, high)", "Karl-Pearson's, Bowley's, Moment-based coefficients"]),
            ("Kurtosis (revisited)", ["beta-2 and gamma-2 — relation and which to use", "Leptokurtic / Mesokurtic / Platykurtic interpretation"]),
        ]
    },
    {
        "label": "Data Preprocessing",
        "file": "Data_Preprocessing_V2.pptx",
        "color": "pink",
        "topics": [
            ("Data Quality", ["Accuracy, completeness, consistency, timeliness, credibility, interpretability", "Sources of dirty data: noisy, incomplete, inconsistent, duplicate"]),
            ("Data Cleaning", ["Handling missing values (ignore tuple, fill manually, global constant, central tendency, regression/classification)", "Handling noisy data: binning (means, medians, boundaries), regression, outlier analysis"]),
            ("Binning (detailed)", ["Equal-frequency bins", "Smoothing by bin means", "Smoothing by bin medians", "Smoothing by bin boundaries"]),
            ("Data Integration", ["Entity identification problem", "Schema integration", "Metadata for attributes"]),
            ("Data Reduction", ["Dimensionality reduction: attribute subset selection, PCA, wavelet transforms", "Stepwise forward/backward selection, decision tree induction", "Numerosity reduction: regression, histograms, clustering, sampling, data cube aggregation", "Data compression: lossless vs lossy"]),
            ("Data Transformation & Discretization", ["Normalization: min-max, z-score, decimal scaling", "Discretization: top-down vs bottom-up, supervised vs unsupervised", "Concept hierarchy generation for nominal & numeric data"]),
            ("Outlier Detection (preprocessing)", ["Z-score method", "Percentile method", "Box plot / Tukey's method (IQR)", "MAD (Median Absolute Deviation) method"]),
        ]
    },
    {
        "label": "Outlier Detection (Advanced)",
        "file": "Outlier_detection_1.pptx",
        "color": "red",
        "topics": [
            ("Introduction to Outliers", ["Definition and causes", "Impact on statistical models and ML algorithms"]),
            ("Types of Outliers", ["Univariate vs multivariate outliers", "Point anomalies", "Contextual anomalies", "Collective anomalies"]),
            ("Statistical Methods", ["Z-score method - formula, threshold, examples", "IQR method - Q1, Q3, bounds, examples"]),
            ("Distance-Based Method: KNN", ["Lazy & non-parametric learning", "Distance metrics (Euclidean, Manhattan, Hamming)", "KNN steps: load data, choose K, calculate distance, sort, assign class", "Outlier identification using KNN"]),
            ("Density-Based Method: LOF", ["K-distance and K-neighbors", "Reachability distance (RD)", "Local reachability density (LRD)", "Local Outlier Factor (LOF) score", "Interpretation: LOF ~1 = inlier; LOF >> 1 = outlier"]),
        ]
    },
    {
        "label": "Feature Engineering",
        "file": "Feature_Engineering.pptx",
        "color": "gray",
        "topics": [
            ("Fundamentals", ["What is a feature?", "What is feature engineering?", "Feature selection vs feature extraction"]),
            ("Curse of Dimensionality", ["Definition and Hughes phenomenon", "Effect on ML algorithm performance", "Data sparsity in high dimensions", "Solutions: cosine similarity, PCA/t-SNE, forward feature selection"]),
            ("Univariate Feature Selection", ["Pearson correlation coefficient (formula, interpretation, when to use)", "Signal-to-noise ratio (S2N)", "Chi-square test (score, degree of freedom, contingency table)", "F-score"]),
            ("Multivariate Feature Selection", ["Filter methods (ANOVA, Pearson correlation, variance thresholding)", "Wrapper methods: forward selection, backward elimination, stepwise selection", "Embedded methods: Lasso, Ridge, Decision Tree"]),
            ("Feature Extraction", ["Definition: mapping F to F'", "Principal Component Analysis (PCA) - overview"]),
        ]
    },
]

# ── Build story ──────────────────────────────────────────────────────────────
story = []

# ── Cover page ───────────────────────────────────────────────────────────────
cover_bg_color = C["purple"]
cover_data = [[Paragraph("FDA Study Guide", cover_title)]]
cover_table = Table(cover_data, colWidths=[W - 36*mm])
cover_table.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,-1), cover_bg_color),
    ("ROUNDEDCORNERS", [8]),
    ("TOPPADDING", (0,0), (-1,-1), 28),
    ("BOTTOMPADDING", (0,0), (-1,-1), 10),
    ("LEFTPADDING", (0,0), (-1,-1), 20),
    ("RIGHTPADDING", (0,0), (-1,-1), 20),
]))
story.append(cover_table)
story.append(Spacer(1, 6))

sub_data = [[Paragraph("Fundamentals of Data Analytics — Complete Topic & Subtopic List", cover_sub)]]
sub_tbl = Table(sub_data, colWidths=[W - 36*mm])
sub_tbl.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,-1), C["purple"]),
    ("TOPPADDING", (0,0), (-1,-1), 6),
    ("BOTTOMPADDING", (0,0), (-1,-1), 22),
    ("LEFTPADDING", (0,0), (-1,-1), 20),
    ("RIGHTPADDING", (0,0), (-1,-1), 20),
]))
story.append(sub_tbl)
story.append(Spacer(1, 22))

# stat pills row
total_topics = sum(len(m["topics"]) for m in modules)
total_subs   = sum(len(s) for m in modules for _, s in m["topics"])
stats = [
    (str(len(modules)), "modules"),
    (str(total_topics), "topics"),
    (str(total_subs), "subtopics"),
]
pill_style_num  = S("Pnum", fontName="Helvetica-Bold", fontSize=22, textColor=C["purple"], alignment=TA_CENTER, leading=26)
pill_style_lbl  = S("Plbl", fontName="Helvetica",      fontSize=10, textColor=C["muted"],  alignment=TA_CENTER, leading=14)
pill_rows = [[Paragraph(n, pill_style_num) for n,_ in stats],
             [Paragraph(l, pill_style_lbl) for _,l in stats]]
pill_tbl = Table(list(zip(*[list(zip(*pill_rows))[i] for i in range(3)])),
                 colWidths=[(W-36*mm)/3]*3)

# simpler: build as two-row table
pill_tbl = Table(
    [[Paragraph(n, pill_style_num) for n,_ in stats],
     [Paragraph(l, pill_style_lbl) for _,l in stats]],
    colWidths=[(W-36*mm)/3]*3
)
pill_tbl.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,-1), C["bg"]),
    ("ROUNDEDCORNERS", [6]),
    ("TOPPADDING", (0,0), (-1,-1), 12),
    ("BOTTOMPADDING", (0,0), (-1,-1), 12),
    ("LINEBELOW", (0,0), (-1,0), 0.3, colors.HexColor("#CCCCCC")),
]))
story.append(pill_tbl)
story.append(Spacer(1, 28))

# ── Module sections ──────────────────────────────────────────────────────────
for m in modules:
    clr = C[m["color"]]
    light_clr = colors.HexColor({
        "purple": "#EEEDFE", "teal": "#E1F5EE", "blue": "#E6F1FB",
        "coral": "#FAECE7", "amber": "#FAEEDA", "green": "#EAF3DE",
        "pink": "#FBEAF0", "red": "#FCEBEB", "gray": "#F1EFE8",
    }[m["color"]])

    # Module header bar
    hdr_data = [[
        Paragraph(m["label"], mod_title),
        Paragraph(m["file"], mod_file),
    ]]
    hdr_tbl = Table(hdr_data, colWidths=[(W-36*mm)*0.72, (W-36*mm)*0.28])
    hdr_tbl.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,-1), clr),
        ("TOPPADDING", (0,0), (-1,-1), 9),
        ("BOTTOMPADDING", (0,0), (-1,-1), 9),
        ("LEFTPADDING", (0,0), (0,-1), 12),
        ("RIGHTPADDING", (-1,0), (-1,-1), 10),
        ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
        ("ALIGN", (1,0), (1,0), "RIGHT"),
    ]))
    story.append(hdr_tbl)

    # Topics
    topic_rows = []
    for topic_name, subs in m["topics"]:
        # topic heading row
        topic_rows.append([
            Paragraph(f"● {topic_name}", S(f"TH{topic_name}", fontName="Helvetica-Bold", fontSize=9.5,
                                            textColor=clr, leading=13, spaceBefore=4, spaceAfter=2)),
        ])
        for s in subs:
            topic_rows.append([
                Paragraph(f"– {s}", S(f"SI{s}", fontName="Helvetica", fontSize=8.5,
                                       textColor=C["muted"], leading=12.5, leftIndent=10)),
            ])

    body_tbl = Table(topic_rows, colWidths=[W - 36*mm])
    body_tbl.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,-1), light_clr),
        ("TOPPADDING", (0,0), (-1,-1), 2),
        ("BOTTOMPADDING", (0,0), (-1,-1), 2),
        ("LEFTPADDING", (0,0), (-1,-1), 12),
        ("RIGHTPADDING", (0,0), (-1,-1), 12),
    ]))
    story.append(body_tbl)
    story.append(Spacer(1, 14))

# ── Footer note ──────────────────────────────────────────────────────────────
story.append(HRFlowable(width="100%", thickness=0.4, color=colors.HexColor("#CCCCCC")))
story.append(Spacer(1, 6))
note_style = S("Note", fontName="Helvetica-Oblique", fontSize=8, textColor=C["muted"], alignment=TA_CENTER)
story.append(Paragraph("Generated from 9 uploaded lecture slides  •  FDA Study Guide", note_style))

doc.build(story)
print("Done!")

ModuleNotFoundError: No module named 'reportlab'